# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [1]:
import os
import re
import json
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login
from pricer.items import Item
from pricer.evaluator import evaluate

In [3]:
LITE_MODE = True

load_dotenv(override=True)

HF_TOKEN = os.environ["HF_TOKEN"]

login(token=HF_TOKEN,add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [5]:
openAI = OpenAI()

#DATA SIZE

In [6]:
train_data = train[:100]
val_data = val[:50]
test_data = test[:50]

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [7]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [8]:
messages_for(train_data[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [9]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [10]:
print(make_jsonl(train_data[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [11]:
def write_jsonl(items, filename):
    with open(filename,'w') as f:
        json_l = make_jsonl(items)
        f.write(json_l)
write_jsonl(train_data[:100], "train.jsonl")
write_jsonl(val_data[:50], "val.jsonl")
write_jsonl(test_data[:50], "test.jsonl")



In [12]:
with open("train.jsonl", "rb") as f:
    train_file = openAI.files.create(
        file=f,
        purpose="fine-tune"
    )

    

In [13]:
print(train_file)

FileObject(id='file-QXXDexEe7A67vGFJPtPrWB', bytes=55120, created_at=1769570958, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


In [14]:
with open("val.jsonl", "rb") as f:
    val_file = openAI.files.create(
        file=f,
        purpose="fine-tune"
    )

print(val_file)

FileObject(id='file-QtzDdm9nT9s6YhKcBPJs9T', bytes=27637, created_at=1769571190, filename='val.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [16]:
openAI.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    hyperparameters={
        "n_epochs": 3,"batch_size": 1
    }
)

FineTuningJob(id='ftjob-1xOzJws8xR7JcG0cPXixAWbr', created_at=1769608584, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=3), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-t8fEmYSzn15PPaucVa1aFJ5N', result_files=[], seed=1393949462, status='validating_files', trained_tokens=None, training_file='file-QXXDexEe7A67vGFJPtPrWB', validation_file='file-QtzDdm9nT9s6YhKcBPJs9T', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=3))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)

In [21]:
openAI.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-1xOzJws8xR7JcG0cPXixAWbr', created_at=1769608584, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=1769609142, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=3), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-t8fEmYSzn15PPaucVa1aFJ5N', result_files=[], seed=1393949462, status='running', trained_tokens=None, training_file='file-QXXDexEe7A67vGFJPtPrWB', validation_file='file-QtzDdm9nT9s6YhKcBPJs9T', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=3))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)], has_more=False, object='list')

In [19]:
job_id = openAI.fine_tuning.jobs.list(limit=1).data[0].id

In [20]:
job_id

'ftjob-1xOzJws8xR7JcG0cPXixAWbr'

# Step 3

Test our fine tuned model

In [27]:
finetuned_model_name = openAI.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [28]:
print(finetuned_model_name)

ft:gpt-4.1-nano-2025-04-14:personal::D30Stc1w


In [29]:
def test_messages(item):
    messages = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": messages}]

In [33]:
def open_source_model(item):
    response = openAI.chat.completions.create(
        model=finetuned_model_name, messages=test_messages(item),
        max_tokens=7)
    return response.choices[0].message.content

    
    

In [34]:
test_messages(test_data[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [ ]:
open_source_model(test_data[0])

'$179.00'

In [36]:
print(test_data[0].price)

219.0


In [39]:
evaluate(open_source_model,test_data,50)

  0%|          | 0/50 [00:00<?, ?it/s]

$20 $6 $25 $10 $6 $65 $371 $39 $11 $285 $473 $60 $30 $17 $10 $11 $31 $12 $120 $64 $16 $56 $28 $135 $163 $225 $81 $10 $160 $65 $70 $4 $85 $10 $14 $36 $1 $1 $64 $35 $167 $44 $21 $130 $89 $10 $19 $1 $101 $34 